# NPTEL Study Notes: Introduction to Hugging Face & Transformers

## 1. The Hugging Face Ecosystem

Hugging Face is the central hub and open-source library for working with transformer-based Large Language Models (LLMs).

* **`transformers` package:** Handles the loading, training, and inference of model architectures (e.g., BERT, GPT).
* **`datasets` package:** Provides access to a vast repository of open-source datasets (e.g., Financial PhraseBank) for fine-tuning.
* **Hugging Face Hub:** A web-based platform hosting pre-trained models, datasets, and tokenizers.

---

## 2. The Standard NLP Pipeline

To process text through a transformer, the data undergoes a specific lifecycle:

1. **Raw Text:** e.g., *"This course is amazing."*
2. **Tokenization:** An algorithm (like Byte Pair Encoding or SentencePiece) breaks text into tokens and maps them to numerical IDs via a dictionary lookup.
3. **Embeddings:** The model's embedding matrix converts token IDs into dense vector representations.
4. **Positional Encoding:** Added to embeddings to retain word order information.
* *Architectural Variations:* GPT-2 uses standard absolute embeddings, while LLaMA uses **RoPE** (Rotary Positional Embedding).


5. **Transformer Blocks:** The vectors pass through multi-head attention, layer normalization, and feed-forward networks.
* *Attention Variations:* Mistral uses **Sliding Window Attention**, while LLaMA uses **Grouped Query Attention (GQA)**.


6. **Logits:** The final output is an unnormalized vector of scores (logits) across the entire vocabulary (e.g., 32,000 tokens).
7. **Softmax:** Converts logits into a probability distribution.

$$P(y_i) = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}$$



*(Where $z_i$ is the logit for token $i$, and $V$ is vocabulary size).*

---

## 3. Tokenization Deep Dive

The `AutoTokenizer` class automatically fetches the exact tokenization rules used during a model's pre-training.

### Key Outputs of a Tokenizer

When `return_tensors="pt"` is set, the tokenizer returns PyTorch tensors containing:

* **`input_ids`:** The sequence of numerical token IDs.
* **`attention_mask`:** A binary mask indicating which tokens the model should attend to (`1`) and which to ignore (`0`). It ignores padding tokens to reduce noise and unnecessary computation. In causal (decoder) models, it also masks future tokens to prevent data leakage.

### Special Tokens (Using BERT/RoBERTa as Examples)

* **`[CLS]` (Classification Token):** Added at the start of a sequence. Its final hidden state is used as the aggregate representation of the whole sentence for classification tasks.
* **`[SEP]` (Separator Token):** Marks the end of a sequence or separates two sentences.
* **`[PAD]` (Padding Token):** Used to make all sequences in a batch the same length (set to 0 in the attention mask).
* **`[UNK]` (Unknown Token):** Handles Out-Of-Vocabulary (OOV) words.

*Note: The transcript contained an error referring to "PTL"; this refers to the example sentence "NPTEL is great."*

---

## 4. Model Classes and Architectures

The `AutoModel` class dynamically loads the correct architecture based on the provided Hugging Face path. You must select the task-specific head:

| Class Name | Use Case | Example Model |
| --- | --- | --- |
| `AutoModel` | Base hidden states only (no task head) | `bert-base-uncased` |
| `AutoModelForSequenceClassification` | Text classification (adds a linear layer head) | `roberta-base-sentiment` |
| `AutoModelForMaskedLM` | Masked Language Modeling (predicting `[MASK]`) | `distilbert-base-uncased` |
| `AutoModelForCausalLM` | Next-token generation (autoregressive) | `gpt2` |

*(Note: The transcript contained auto-generated transcription errors such as "automobile model" and "math scale" which actually refer to `AutoModel` and `Masked LM` respectively).*

### Accessing Model Internals

By default, models output loss (if labels are provided) and logits. By inspecting the model output object, you can access internal representations:

* **`output_hidden_states=True`:** Returns the hidden states for every layer.
* **`output_attentions=True`:** Returns the attention weight matrices for every head in every layer (useful for plotting heatmaps to see what the model focuses on).

---

## 5. Fine-Tuning Models

Fine-tuning updates a pre-trained model's weights on a downstream dataset (like Financial PhraseBank).

### Method 1: Standard PyTorch Training Loop

Requires manual iteration over batches:

1. `optimizer.zero_grad()`: Clears old gradients.
2. `loss = model(inputs).loss`: Computes the task loss (e.g., Cross-Entropy).
3. `loss.backward()`: Computes gradients via backpropagation.
4. `optimizer.step()`: Updates the model's weights (e.g., using Stochastic Gradient Descent).

### Method 2: Hugging Face `Trainer` API

An abstraction that drastically simplifies fine-tuning.

* **`TrainingArguments`:** Defines hyperparameters (batch size, learning rate, epochs, evaluation strategy).
* **`Trainer` Class:** Takes the model, arguments, train/eval datasets, and a `compute_metrics` function.
* **Callbacks:** Can be added for dynamic training control (e.g., `EarlyStoppingCallback` to halt training if validation loss degrades).

---

## 6. Text Generation (Causal LM)

For decoder-only models (like GPT-2), generation relies on the `model.generate()` method. The behavior is controlled by decoding parameters:

* **Greedy Decoding:** Always picks the highest probability token (`do_sample=False`). *(Note: The transcript incorrectly transcribed this as "grid decoding")*.
* **Sampling (`do_sample=True`):** Picks the next token probabilistically.
* **Top-$p$ (Nucleus Sampling):** Samples only from the smallest set of tokens whose cumulative probability exceeds $p$ (e.g., `top_p=0.9`).
* **Top-$k$ Sampling:** Samples only from the $k$ most likely next tokens.
* **Temperature:** Adjusts the sharpness of the probability distribution. A lower temperature (e.g., 0.3) makes the model more deterministic, while a higher temperature increases randomness.

---

## 7. Hugging Face Pipelines

The `pipeline()` function is a high-level abstraction that wraps tokenization, model inference, and post-processing into a single line of code.

* **Example:** `pipeline("sentiment-analysis", model="roberta-base")`
* **Use Case:** Ideal for rapid prototyping and testing off-the-shelf tasks. Not typically used for custom research or complex fine-tuning.

---

## Practice Questions

**Q1. When preparing a batch of sentences of varying lengths for a transformer model, which of the following is strictly required to prevent the model from processing meaningless data?**
a) Setting `output_hidden_states=True`
b) Utilizing a `[CLS]` token at the end of the sequence
c) Applying an `attention_mask` with 0s for padding tokens
d) Using Rotary Positional Embeddings (RoPE)
**Answer:** c) Applying an `attention_mask` with 0s for padding tokens. Explanation: The attention mask tells the self-attention mechanism to ignore padding tokens, saving computation and reducing noise.

**Q2. If you want to train a BERT model to predict a missing word in a sentence, which Hugging Face class should you use?**
a) `AutoModelForCausalLM`
b) `AutoModelForSequenceClassification`
c) `AutoModelForMaskedLM`
d) `AutoTokenizer`
**Answer:** c) `AutoModelForMaskedLM`. Explanation: BERT is an encoder-only model designed for Masked Language Modeling (MLM). Causal LM is for decoder-only autoregressive models.

**Q3. During a custom PyTorch training loop, what is the exact purpose of `loss.backward()`?**
a) It updates the model weights based on the learning rate.
b) It clears the gradients from the previous epoch.
c) It calculates the Cross-Entropy loss between logits and labels.
d) It performs backpropagation to populate the gradient fields of the tensors.
**Answer:** d). Explanation: `loss.backward()` computes gradients. Weight updating is handled by `optimizer.step()`, and clearing gradients is handled by `optimizer.zero_grad()`.

---

## Revision Summary

* **Hub & Libraries:** `transformers` (models), `datasets` (data).
* **Tokenization:** Converts strings to IDs (`input_ids`) and tracks real vs. padded data (`attention_mask`). Requires handling special tokens (`[CLS]`, `[SEP]`).
* **Auto Classes:** Use `AutoModelFor[Task]` to instantly load the correct architecture and output head.
* **Training:** Can be done manually in PyTorch (`zero_grad` $\rightarrow$ `backward` $\rightarrow$ `step`) or via the automated `Trainer` API.
* **Generation Strategy:** Control creativity in Decoder models using `do_sample`, `top_p`, `top_k`, and `temperature`.